# Pricing engine — price and Greeks vs spot (European vs American)

Visual demonstration of the pricing engine. Percent `# %%` cells: open as a notebook or run as
a script. It ONLY calls the tested pricing engine and plots; no pricing logic lives here.

Run: `uv run --group notebooks python notebooks/demo_pricing_greeks.py`

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from algotrading.infra.pricing import from_spot, price

Build pricing states directly from spot and sweep across spot. The tested pricing engine
takes a ``PricingState`` (built via ``from_spot``) and returns price + Greeks; European is
analytic (Black-76), American is the lattice. ``option_right`` is ``"C"``/``"P"``; the
discount factor encodes the 4% rate over the 0.5y maturity.

In [ ]:
spots = np.linspace(70.0, 130.0, 60)
_DISCOUNT = float(np.exp(-0.04 * 0.5))  # rate 4% over maturity 0.5y


def _curve(style: str, attr: str) -> np.ndarray:
    results = [
        price(
            from_spot(
                spot=float(s),
                strike=100.0,
                maturity_years=0.5,
                volatility=0.25,
                discount_factor=_DISCOUNT,
                option_right="C",
                carry=0.0,
                exercise_style=style,
            )
        )
        for s in spots
    ]
    return np.array([getattr(r, attr) for r in results])

Price and the first/second-order Greeks across spot, European (analytic) vs American (lattice).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for attr, ax in zip(["price", "delta", "gamma", "vega"], axes.flat, strict=True):
    ax.plot(spots, _curve("european", attr), color="C0", label="European")
    ax.plot(spots, _curve("american", attr), color="C1", ls="--", label="American")
    ax.axvline(100.0, color="grey", lw=0.8, ls=":")  # strike
    ax.set(xlabel="spot", ylabel=attr, title=f"{attr} vs spot")
    ax.legend()
fig.tight_layout()
plt.show()